# preprocessing 

Input: the official `UNSW_NB15_training-set.csv`. Output: a stratified **train / validation** split, a fitted scaler and metadata.

**Model inputs (20 columns, about 39 after one-hot):**
- Direct from Zeek: `dur`, `proto`, `service`, `state`, `sbytes`, `dbytes`, `spkts`, `dpkts`
- Derived: `sload`, `dload`, `smean`, `dmean` (recomputed here), plus `is_sm_ips_ports` and 7 `ct_*` features taken from the file

In [16]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

In [17]:
CSV_PATH = "UNSW_NB15_training-set.csv"   # or a full path, e.g. r"C:\Users\YourName\Downloads\UNSW_NB15_training-set.csv"
OUT_DIR = "processed"

RANDOM_STATE = 42
VAL_SIZE = 0.20             # 80-20
DROP_DUPLICATES = False     # keep duplicates: repeated flows are normal in floods (DoS, Generic)
GROUP_DUPLICATES = True     # identical rows always go to the SAME split, so no train/validation leakage
USE_STATE = True            # set False if you decide the state mapping is too shaky
RECOMPUTE_DERIVED = True    # recompute sload/dload/smean/dmean so live == training

In [18]:
REQUIRED_COLUMNS = [
    "dur", "proto", "service", "state", "sbytes", "dbytes", "spkts", "dpkts",
    "sload", "dload", "smean", "dmean", "is_sm_ips_ports",
    "ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm",
    "ct_src_dport_ltm", "ct_dst_sport_ltm", "ct_dst_src_ltm",
    "attack_cat", "label",
]
CT_COLS = ["ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm",
           "ct_src_dport_ltm", "ct_dst_sport_ltm", "ct_dst_src_ltm"]
NUMERIC_COLS = ["dur", "sbytes", "dbytes", "spkts", "dpkts",
                "sload", "dload", "smean", "dmean", "is_sm_ips_ports", *CT_COLS]
LOG_COLS = ["dur", "sbytes", "dbytes", "spkts", "dpkts", "sload", "dload", "smean", "dmean"]
NO_SCALE = ["is_sm_ips_ports"]

PROTO_CATS = ["tcp", "udp", "icmp", "other"]
SERVICE_CATS = ["-", "dns", "http", "smtp", "ftp", "ftp-data", "ssh",
                "pop3", "ssl", "irc", "dhcp", "snmp", "radius", "other"]
STATE_GROUPS = ["ok", "no_reply", "reset", "other"]
ARGUS_STATE_MAP = {"FIN": "ok", "CLO": "ok", "CON": "ok", "ACC": "ok",
                   "INT": "no_reply", "REQ": "no_reply", "RST": "reset"}
ZEEK_STATE_MAP = {"SF": "ok", "S1": "ok", "S2": "ok", "S3": "ok", "S0": "no_reply",
                  "REJ": "reset", "RSTO": "reset", "RSTR": "reset",
                  "RSTOS0": "reset", "RSTRH": "reset"}
CAT_SPEC = {"proto": PROTO_CATS, "service": SERVICE_CATS, "state": STATE_GROUPS}

CLASS_NAMES = ["Normal", "DoS", "Reconnaissance", "Fuzzers", "Exploits", "Generic", "Other"]
KEEP_ATTACKS = ["DoS", "Reconnaissance", "Fuzzers", "Exploits", "Generic"]  # the rest become "Other"

In [19]:
def load_training_csv(path):
    df = pd.read_csv(path, low_memory=False, encoding="latin-1")
    df.columns = [c.strip().lower() for c in df.columns]
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}\nColumns found: {list(df.columns)}")
    print(f"loaded {len(df):,} rows, {df.shape[1]} columns")
    return df


def group_proto(s):
    s = s.fillna("").astype(str).str.strip().str.lower()
    return s.where(s.isin(PROTO_CATS[:-1]), "other")


def group_service(s):
    s = s.fillna("-").astype(str).str.strip().str.lower().replace("", "-")
    return s.where(s.isin(SERVICE_CATS[:-1]), "other")


def group_state(s, state_map):
    s = s.fillna("").astype(str).str.strip().str.upper()
    return s.map(state_map).fillna("other")


def safe_div(a, b):
    a = np.asarray(a, dtype="float64")
    b = np.asarray(b, dtype="float64")
    out = np.zeros_like(a)
    np.divide(a, b, out=out, where=b > 0)
    return out


def clean_and_engineer(df, state_map, recompute=RECOMPUTE_DERIVED):
    df = df.copy()
    for c in NUMERIC_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    df["label"] = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int)

    df["proto"] = group_proto(df["proto"])
    df["service"] = group_service(df["service"])
    df["state"] = group_state(df["state"], state_map)

    if recompute:
        df["sload"] = safe_div(df["sbytes"] * 8, df["dur"])
        df["dload"] = safe_div(df["dbytes"] * 8, df["dur"])
        df["smean"] = safe_div(df["sbytes"], df["spkts"])
        df["dmean"] = safe_div(df["dbytes"], df["dpkts"])

    df["is_sm_ips_ports"] = (df["is_sm_ips_ports"] > 0).astype(int)
    for c in CT_COLS:
        df[c] = df[c].clip(lower=0)
    return df


def build_targets(df):
    cat = (df["attack_cat"].fillna("Normal").astype(str).str.strip()
           .replace({"Backdoor": "Backdoors", "": "Normal"}))
    group = cat.where(cat.isin(["Normal", *KEEP_ATTACKS]), "Other")
    group = group.mask(df["label"] == 0, "Normal")
    group = group.mask((df["label"] == 1) & (group == "Normal"), "Other")
    df = df.copy()
    df["attack_group"] = group
    df["y_multi"] = group.map({c: i for i, c in enumerate(CLASS_NAMES)}).astype(int)
    return df


def build_model_frame(df, use_state=USE_STATE):
    num = df[NUMERIC_COLS].astype("float64").copy()
    for c in LOG_COLS:
        num[c] = np.log1p(num[c].clip(lower=0))
    parts = [num]
    for col, cats in CAT_SPEC.items():
        if col == "state" and not use_state:
            continue
        cat = pd.Series(pd.Categorical(df[col], categories=cats), index=df.index)
        parts.append(pd.get_dummies(cat, prefix=col).astype("float64"))
    return pd.concat(parts, axis=1)


def scale_columns(frame):
    return [c for c in frame.columns if c in NUMERIC_COLS and c not in NO_SCALE]

In [20]:
raw = load_training_csv(CSV_PATH)
raw.head()

loaded 175,341 rows, 45 columns


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


In [21]:
df = clean_and_engineer(raw, ARGUS_STATE_MAP)
df = build_targets(df)
frame = build_model_frame(df)
y_bin = df["label"].to_numpy()
y_multi = df["y_multi"].to_numpy()

print("feature columns:", frame.shape[1])
print(df["attack_group"].value_counts())

feature columns: 39
attack_group
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Other              5009
Name: count, dtype: int64


In [22]:
if DROP_DUPLICATES:
    key = frame.copy()
    key["__y"] = y_multi
    keep = ~key.duplicated().to_numpy()
    print(f"dropping {(~keep).sum():,} duplicate rows")
    frame = frame[keep].reset_index(drop=True)
    y_bin, y_multi = y_bin[keep], y_multi[keep]
print("rows kept:", len(frame))

rows kept: 175341


In [23]:
from sklearn.model_selection import StratifiedGroupKFold

idx = np.arange(len(frame))
if GROUP_DUPLICATES:
    key_df = frame.copy()
    key_df["__y"] = y_multi
    groups = key_df.groupby(list(key_df.columns), sort=False).ngroup().to_numpy()
    print(f"{len(frame):,} rows form {groups.max() + 1:,} distinct groups")
    sgkf = StratifiedGroupKFold(n_splits=round(1 / VAL_SIZE), shuffle=True, random_state=RANDOM_STATE)
    idx_train, idx_val = next(sgkf.split(frame, y_multi, groups))
else:
    idx_train, idx_val = train_test_split(
        idx, test_size=VAL_SIZE, stratify=y_multi, random_state=RANDOM_STATE)

print(f"train: {len(idx_train):,}   validation: {len(idx_val):,}  ({100 * len(idx_val) / len(idx):.1f}% validation)")

share = pd.DataFrame({
    "all": pd.Series(y_multi).value_counts(normalize=True),
    "train": pd.Series(y_multi[idx_train]).value_counts(normalize=True),
    "val": pd.Series(y_multi[idx_val]).value_counts(normalize=True),
}).sort_index()
share.index = [CLASS_NAMES[i] for i in share.index]
(share * 100).round(2)

175,341 rows form 104,608 distinct groups
train: 140,272   validation: 35,069  (20.0% validation)


,all,train,val
Normal,31.94,31.94,31.94
DoS,6.99,6.99,6.99
Reconnaissance,5.98,5.98,5.99
Fuzzers,10.37,10.37,10.37
Exploits,19.04,19.05,19.04
Generic,22.81,22.81,22.81
Other,2.86,2.86,2.86


In [24]:
cols = scale_columns(frame)
scaler = StandardScaler().fit(frame.iloc[idx_train][cols])
scaled = frame.copy()
scaled[cols] = scaler.transform(frame[cols])
X = scaled.to_numpy(dtype="float32")

splits = {"train": idx_train, "val": idx_val}
arrays = {}
for name, ix in splits.items():
    arrays[f"X_{name}"] = X[ix]
    arrays[f"y_bin_{name}"] = y_bin[ix].astype("int8")
    arrays[f"y_multi_{name}"] = y_multi[ix].astype("int8")

In [25]:
out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)

np.savez_compressed(out / "unsw_nb15_processed.npz", **arrays)
joblib.dump(scaler, out / "scaler.joblib")

cw_multi = compute_class_weight("balanced", classes=np.arange(len(CLASS_NAMES)), y=y_multi[idx_train])
cw_bin = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_bin[idx_train])
meta = {
    "feature_names": list(frame.columns),
    "scale_cols": cols,
    "class_names": CLASS_NAMES,
    "class_weight_multi": {int(i): float(w) for i, w in enumerate(cw_multi)},
    "class_weight_binary": {0: float(cw_bin[0]), 1: float(cw_bin[1])},
    "config": {"use_state": USE_STATE, "recompute_derived": RECOMPUTE_DERIVED,
               "drop_duplicates": DROP_DUPLICATES, "group_duplicates": GROUP_DUPLICATES,
               "random_state": RANDOM_STATE, "val_size": VAL_SIZE},
    "zeek_state_map": ZEEK_STATE_MAP,
}
(out / "meta.json").write_text(json.dumps(meta, indent=2))
print("saved to", out.resolve())

saved to D:\College\THAPAR\SEM 5\Machine Learning\Network Project\Network_Intrusion_Detection_System\processed


In [26]:
class_lookup = dict(enumerate(CLASS_NAMES))
for name, ix in splits.items():
    part = pd.DataFrame(X[ix], columns=frame.columns)
    part["label"] = y_bin[ix]
    part["attack_class_id"] = y_multi[ix]
    part["attack_class"] = pd.Series(y_multi[ix]).map(class_lookup).to_numpy()
    fname = "train.csv" if name == "train" else "validation.csv"
    part.to_csv(out / fname, index=False)
    print(f"{fname:16s} {part.shape[0]:>9,} rows x {part.shape[1]} columns")

train.csv          140,272 rows x 42 columns
validation.csv      35,069 rows x 42 columns


## 12. Summary of the split

In [27]:
for name, ix in splits.items():
    counts = pd.Series(y_multi[ix]).map(dict(enumerate(CLASS_NAMES))).value_counts()
    print(f"\n{name}: {len(ix):,} rows, {100 * y_bin[ix].mean():.1f}% attack")
    print(counts.to_string())


train: 140,272 rows, 68.1% attack
Normal            44800
Generic           32000
Exploits          26715
Fuzzers           14547
DoS                9811
Reconnaissance     8392
Other              4007

val: 35,069 rows, 68.1% attack
Normal            11200
Generic            8000
Exploits           6678
Fuzzers            3637
DoS                2453
Reconnaissance     2099
Other              1002


## 13. Feature selection (GA, PSO, mutual-information filter)

All three methods search for a good subset of the 39 features using **training data only**:
the model is fitted on a train subsample and scored on a validation subsample (macro F1, with a small penalty for large subsets).
The saved `.npz` / CSV files above still contain all 39 features; the chosen subset is stored in `meta.json` as `selected_features`.

**Note:** because the search tunes against the validation set, the validation score of the selected subset is slightly optimistic. Confirm on a separate test set or on your own Zeek captures.

In [28]:
# ---- shared setup for feature selection ----
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.feature_selection import mutual_info_classif

FS_SEED = RANDOM_STATE
FS_TRAIN_ROWS = 20_000      # subsample used inside the search (keeps it fast)
FS_VAL_ROWS = 10_000
PENALTY = 0.01              # fitness = macro_f1 - PENALTY * (n_selected / n_features)

Xtr_full, ytr_full = arrays["X_train"], arrays["y_multi_train"]
Xva_full, yva_full = arrays["X_val"], arrays["y_multi_val"]
feature_names = list(frame.columns)
n_feat = Xtr_full.shape[1]

def stratified_subsample(X, y, n, seed):
    if n >= len(y):
        return X, y
    ix, _ = train_test_split(np.arange(len(y)), train_size=n, stratify=y, random_state=seed)
    return X[ix], y[ix]

Xtr_s, ytr_s = stratified_subsample(Xtr_full, ytr_full, FS_TRAIN_ROWS, FS_SEED)
Xva_s, yva_s = stratified_subsample(Xva_full, yva_full, FS_VAL_ROWS, FS_SEED)

_cache = {}
def fitness(mask):
    """Higher is better. mask: boolean array of length n_feat."""
    mask = np.asarray(mask, dtype=bool)
    if not mask.any():
        return 0.0
    key = mask.tobytes()
    if key not in _cache:
        clf = DecisionTreeClassifier(max_depth=12, class_weight="balanced", random_state=FS_SEED)
        clf.fit(Xtr_s[:, mask], ytr_s)
        f1 = f1_score(yva_s, clf.predict(Xva_s[:, mask]), average="macro")
        _cache[key] = f1 - PENALTY * mask.sum() / n_feat
    return _cache[key]

baseline_fit = fitness(np.ones(n_feat, dtype=bool))
print(f"all {n_feat} features -> fitness {baseline_fit:.4f}")

all 39 features -> fitness 0.6565


In [29]:
### Method 1: Genetic Algorithm (GA)
def run_ga(pop_size=30, generations=15, cx_prob=0.8, mut_prob=0.03, tourn=3, elite=2, seed=FS_SEED):
    rng = np.random.default_rng(seed)
    pop = rng.random((pop_size, n_feat)) < 0.5
    pop[0] = True                                     # include the full feature set
    history = []
    for g in range(generations):
        fit = np.array([fitness(ind) for ind in pop])
        history.append(fit.max())
        order = np.argsort(-fit)
        new_pop = [pop[i].copy() for i in order[:elite]]          # elitism
        while len(new_pop) < pop_size:
            # tournament selection
            p1 = pop[max(rng.choice(pop_size, tourn, replace=False), key=lambda i: fit[i])]
            p2 = pop[max(rng.choice(pop_size, tourn, replace=False), key=lambda i: fit[i])]
            # uniform crossover
            if rng.random() < cx_prob:
                take = rng.random(n_feat) < 0.5
                child = np.where(take, p1, p2)
            else:
                child = p1.copy()
            # bit-flip mutation
            flip = rng.random(n_feat) < mut_prob
            child = np.logical_xor(child, flip)
            if not child.any():
                child[rng.integers(n_feat)] = True
            new_pop.append(child)
        pop = np.array(new_pop)
        print(f"GA gen {g + 1:2d}/{generations}  best fitness {history[-1]:.4f}")
    fit = np.array([fitness(ind) for ind in pop])
    best = pop[fit.argmax()]
    return best, fit.max(), history

t0 = time.time()
ga_mask, ga_fit, ga_hist = run_ga()
print(f"GA kept {ga_mask.sum()}/{n_feat} features, fitness {ga_fit:.4f}, {time.time() - t0:.0f}s")

GA gen  1/15  best fitness 0.6791
GA gen  2/15  best fitness 0.6807
GA gen  3/15  best fitness 0.6807
GA gen  4/15  best fitness 0.6807
GA gen  5/15  best fitness 0.6878
GA gen  6/15  best fitness 0.6887
GA gen  7/15  best fitness 0.6887
GA gen  8/15  best fitness 0.6891
GA gen  9/15  best fitness 0.6898
GA gen 10/15  best fitness 0.6898
GA gen 11/15  best fitness 0.6901
GA gen 12/15  best fitness 0.6901
GA gen 13/15  best fitness 0.6901
GA gen 14/15  best fitness 0.6903
GA gen 15/15  best fitness 0.6903
GA kept 17/39 features, fitness 0.6903, 18s


In [30]:
### Method 2: Binary Particle Swarm Optimization (PSO)
def run_pso(n_particles=30, iters=15, w=0.7, c1=1.5, c2=1.5, vmax=4.0, seed=FS_SEED):
    rng = np.random.default_rng(seed)
    pos = rng.random((n_particles, n_feat)) < 0.5
    pos[0] = True
    vel = rng.uniform(-1, 1, (n_particles, n_feat))
    pbest = pos.copy()
    pbest_fit = np.array([fitness(p) for p in pos])
    g = pbest_fit.argmax()
    gbest, gbest_fit = pbest[g].copy(), pbest_fit[g]
    history = [gbest_fit]
    for it in range(iters):
        r1 = rng.random((n_particles, n_feat))
        r2 = rng.random((n_particles, n_feat))
        vel = (w * vel
               + c1 * r1 * (pbest.astype(float) - pos.astype(float))
               + c2 * r2 * (gbest.astype(float) - pos.astype(float)))
        vel = np.clip(vel, -vmax, vmax)
        prob = 1.0 / (1.0 + np.exp(-vel))             # sigmoid -> probability of bit = 1
        pos = rng.random((n_particles, n_feat)) < prob
        for i in range(n_particles):
            if not pos[i].any():
                pos[i, rng.integers(n_feat)] = True
        fit = np.array([fitness(p) for p in pos])
        better = fit > pbest_fit
        pbest[better], pbest_fit[better] = pos[better], fit[better]
        if pbest_fit.max() > gbest_fit:
            g = pbest_fit.argmax()
            gbest, gbest_fit = pbest[g].copy(), pbest_fit[g]
        history.append(gbest_fit)
        print(f"PSO iter {it + 1:2d}/{iters}  best fitness {gbest_fit:.4f}")
    return gbest, gbest_fit, history

t0 = time.time()
pso_mask, pso_fit, pso_hist = run_pso()
print(f"PSO kept {pso_mask.sum()}/{n_feat} features, fitness {pso_fit:.4f}, {time.time() - t0:.0f}s")

PSO iter  1/15  best fitness 0.6791
PSO iter  2/15  best fitness 0.6820
PSO iter  3/15  best fitness 0.6820
PSO iter  4/15  best fitness 0.6820
PSO iter  5/15  best fitness 0.6820
PSO iter  6/15  best fitness 0.6820
PSO iter  7/15  best fitness 0.6820
PSO iter  8/15  best fitness 0.6820
PSO iter  9/15  best fitness 0.6820
PSO iter 10/15  best fitness 0.6820
PSO iter 11/15  best fitness 0.6820
PSO iter 12/15  best fitness 0.6820
PSO iter 13/15  best fitness 0.6820
PSO iter 14/15  best fitness 0.6820
PSO iter 15/15  best fitness 0.6820
PSO kept 20/39 features, fitness 0.6820, 35s


In [31]:
### Method 3: Mutual-information filter (simple non-metaheuristic baseline)

##Ranks features by mutual information with the attack class, then picks the top-k that gives the best fitness.

t0 = time.time()
mi = mutual_info_classif(Xtr_s, ytr_s, random_state=FS_SEED)
rank = np.argsort(-mi)

best_k, mi_fit, mi_mask = None, -1.0, None
for k in range(5, n_feat + 1):
    m = np.zeros(n_feat, dtype=bool)
    m[rank[:k]] = True
    f = fitness(m)
    if f > mi_fit:
        best_k, mi_fit, mi_mask = k, f, m
print(f"MI filter kept top {best_k}/{n_feat} features, fitness {mi_fit:.4f}, {time.time() - t0:.0f}s")
print("top 10 by MI:", [feature_names[i] for i in rank[:10]])

MI filter kept top 8/39 features, fitness 0.6625, 7s
top 10 by MI: ['sbytes', 'smean', 'sload', 'dbytes', 'dmean', 'ct_dst_sport_ltm', 'dur', 'ct_srv_dst', 'dload', 'ct_srv_src']


In [32]:
### Compare all subsets and save the chosen one

##Each subset is evaluated with a Random Forest trained on the **full** training set and scored on the full validation set.

candidates = {
    "all_features": np.ones(n_feat, dtype=bool),
    "GA": ga_mask,
    "PSO": pso_mask,
    "MI_filter": mi_mask,
}

rows = []
for name, mask in candidates.items():
    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                n_jobs=-1, random_state=FS_SEED)
    rf.fit(Xtr_full[:, mask], ytr_full)
    pred = rf.predict(Xva_full[:, mask])
    rows.append({
        "method": name,
        "n_features": int(mask.sum()),
        "macro_F1": round(f1_score(yva_full, pred, average="macro"), 4),
        "weighted_F1": round(f1_score(yva_full, pred, average="weighted"), 4),
        "seconds": round(time.time() - t0),
    })
comparison = pd.DataFrame(rows).set_index("method")
comparison

,n_features,macro_F1,weighted_F1,seconds
method,,,,
all_features,39,0.6224,0.7795,4
GA,17,0.6983,0.8115,2
PSO,20,0.6875,0.8078,2
MI_filter,8,0.6446,0.7887,2


In [33]:
# Pick the subset with the best macro F1; on a tie prefer fewer features.
best_method = comparison.sort_values(["macro_F1", "n_features"], ascending=[False, True]).index[0]
best_mask = candidates[best_method]
selected_features = [f for f, keep in zip(feature_names, best_mask) if keep]
dropped = [f for f, keep in zip(feature_names, best_mask) if not keep]
print(f"chosen: {best_method}  ({len(selected_features)} features)")
print("dropped:", dropped)

# Which of the risky live-pipeline features survived?
risky = ["is_sm_ips_ports", *CT_COLS]
print("ct_* / is_sm_ips_ports kept:", [f for f in risky if f in selected_features])

meta["selected_features"] = selected_features
meta["feature_selection"] = {
    "chosen_method": best_method,
    "seed": FS_SEED,
    "penalty": PENALTY,
    "subsample_rows": {"train": FS_TRAIN_ROWS, "val": FS_VAL_ROWS},
    "results": comparison.reset_index().to_dict(orient="records"),
    "subsets": {name: [f for f, k in zip(feature_names, m) if k] for name, m in candidates.items()},
}
(out / "meta.json").write_text(json.dumps(meta, indent=2))
print("meta.json updated with selected_features")

chosen: GA  (17 features)
dropped: ['dur', 'dbytes', 'dpkts', 'sload', 'dload', 'smean', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'proto_tcp', 'proto_icmp', 'proto_other', 'service_smtp', 'service_pop3', 'service_ssl', 'service_irc', 'service_snmp', 'service_other', 'state_reset', 'state_other']
ct_* / is_sm_ips_ports kept: ['is_sm_ips_ports', 'ct_srv_src', 'ct_srv_dst']
meta.json updated with selected_features


## 14. Live helper (use later in the Zeek pipeline)

Takes an unscaled model frame built with `build_model_frame()`, aligns it to the training columns, applies the saved scaler, and (if feature selection was run) keeps only `selected_features`.

In [34]:
def transform_with_artifacts(frame, artifacts_dir=OUT_DIR, use_selected=True):
    art = Path(artifacts_dir)
    scaler = joblib.load(art / "scaler.joblib")
    meta = json.loads((art / "meta.json").read_text())
    frame = frame.reindex(columns=meta["feature_names"], fill_value=0.0).copy()
    frame[meta["scale_cols"]] = scaler.transform(frame[meta["scale_cols"]])
    if use_selected and meta.get("selected_features"):
        frame = frame[meta["selected_features"]]
    return frame.to_numpy(dtype="float32")

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

proc = Path("processed")
meta = json.loads((proc / "meta.json").read_text())
old = meta["class_names"]
assert old == ["Normal", "DoS", "Reconnaissance", "Fuzzers", "Exploits", "Generic", "Other"], \
    f"Expected the 7-class labels, found {old}. Was this cell already run?"

new = ["Normal", "Reconnaissance", "Fuzzers", "Generic", "Exploit-like"]
to_new = {"Normal": "Normal", "DoS": "Exploit-like", "Exploits": "Exploit-like", "Other": "Exploit-like",
          "Reconnaissance": "Reconnaissance", "Fuzzers": "Fuzzers", "Generic": "Generic"}
lut = np.array([new.index(to_new[n]) for n in old], dtype="int8")      # old class index -> new class index


def remap_npz(path):
    with np.load(path) as f:                                            # closed before the file is rewritten
        arrays = {k: f[k] for k in f.files}
    for k in arrays:
        if k.startswith("y_multi"):
            arrays[k] = lut[arrays[k].astype(int)]
    np.savez_compressed(path, **arrays)
    return arrays


train_arrays = remap_npz(proc / "unsw_nb15_processed.npz")
if (proc / "unsw_nb15_test.npz").exists():
    remap_npz(proc / "unsw_nb15_test.npz")

for name in ("train.csv", "validation.csv"):
    p = proc / name
    if p.exists():
        df = pd.read_csv(p)
        if "attack_class_id" in df.columns:
            df["attack_class_id"] = lut[df["attack_class_id"].astype(int)]
            df["attack_class"] = df["attack_class_id"].map(dict(enumerate(new)))
            df.to_csv(p, index=False)

y = train_arrays["y_multi_train"].astype(int)
cw = len(y) / (len(new) * np.bincount(y, minlength=len(new)))
meta["class_names"] = new
meta["class_weight_multi"] = {str(i): float(w) for i, w in enumerate(cw)}
meta.setdefault("config", {})["class_scheme"] = "merged"
(proc / "meta.json").write_text(json.dumps(meta, indent=2))
print("training rows per class:", dict(zip(new, np.bincount(y, minlength=len(new)).tolist())))

training rows per class: {'Normal': 44800, 'Reconnaissance': 8392, 'Fuzzers': 14547, 'Generic': 32000, 'Exploit-like': 40533}
